In [3]:
# Aniesh Metric
import numpy as np
def compute_accuracy_from_confusion_matrix(cm):
    # True Positives (TP): Diagonal elements
    true_positive = np.diag(cm)

    # False Positives (FP): Column sum - TP
    false_positive = cm.sum(axis=0) - true_positive

    # False Negatives (FN): Row sum - TP
    false_negative = cm.sum(axis=1) - true_positive

    # True Negatives (TN): Total sum - (TP + FP + FN)
    total = cm.sum()
    true_negative = total - (true_positive + false_positive + false_negative)

    # Debug prints for each class
    for i in range(len(true_positive)):

        if true_negative[i] + false_positive[i] == 0:
            print(f"  ⚠️ No negative examples for class {i}")
        if true_positive[i] + false_negative[i] == 0:
            print(f"  ⚠️ No positive examples for class {i}")
        if true_negative[i] == 0 and false_positive[i] > 0:
            print(f"Class {i}:")
            print(f"  TP = {true_positive[i]}, FP = {false_positive[i]}, FN = {false_negative[i]}, TN = {true_negative[i]}")
            print(f"  ⚠️ Specificity for class {i} is 0 — all negative samples predicted as class {i}")
            print("\nFull Confusion Matrix:\n", cm)

    # Accuracy: Total correct predictions / Total samples
    accuracy = true_positive.sum() / total

    # Sensitivity (Recall): TP / (TP + FN)
    with np.errstate(divide='ignore', invalid='ignore'):
        sensitivity = np.divide(true_positive, true_positive + false_negative)
        sensitivity[np.isnan(sensitivity)] = 0

    # Specificity: TN / (TN + FP)
    with np.errstate(divide='ignore', invalid='ignore'):
        specificity = np.divide(true_negative, true_negative + false_positive)
        specificity[np.isnan(specificity)] = 0

    # Balanced Accuracy: Average of Sensitivity and Specificity
    balanced_accuracy = np.mean((sensitivity + specificity) / 2)

    return accuracy, balanced_accuracy

def compute_additional_metrics_from_confusion_matrix(cm):
    """
    Compute specificity, sensitivity, positive predictive value (PPV),
    and negative predictive value (NPV) from a multi-class confusion matrix.

    :param cm: NumPy array representing the confusion matrix (square matrix).
    :return: Dictionary containing computed metrics for each class.
    """
    # True Positives (TP): Diagonal elements
    true_positive = np.diag(cm)

    # False Positives (FP): Column sum - TP
    false_positive = cm.sum(axis=0) - true_positive

    # False Negatives (FN): Row sum - TP
    false_negative = cm.sum(axis=1) - true_positive

    # True Negatives (TN): Total sum - (TP + FP + FN)
    total = cm.sum()
    true_negative = total - (true_positive + false_positive + false_negative)

    # Compute metrics
    sensitivity = true_positive / (true_positive + false_negative)  # Recall
    specificity = true_negative / (true_negative + false_positive)
    ppv = true_positive / (true_positive + false_positive)  # Precision
    npv = true_negative / (true_negative + false_negative)
    f1_score = 2 * ppv * sensitivity / (ppv + sensitivity)

    # Handle division by zero
    sensitivity = np.nan_to_num(sensitivity)
    specificity = np.nan_to_num(specificity)
    ppv = np.nan_to_num(ppv)
    npv = np.nan_to_num(npv)
    f1_score = np.nan_to_num(f1_score)

    # Return metrics for each class
    return {
        "sensitivity": sensitivity.tolist(),
        "specificity": specificity.tolist(),
        "positive_predictive_value": ppv.tolist(),
        "negative_predictive_value": npv.tolist(),
        "f1_score": f1_score.tolist(),
    }


In [41]:
import pandas as pd
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

# Load data
path = '/niddk-data-central/leo_workspace/iWatch-Validation/W/CHAP-FT/predictions/i0386A.csv' # TODO: Use your path
df = pd.read_csv(path)

# Ensure the order and mapping
class_names = sorted(set(df['prediction']).union(set(df['label'])))
class_to_int = {name: i for i, name in enumerate(class_names)}

# Map to integer labels
y_pred = df['prediction'].map(class_to_int).values
y_true = df['label'].map(class_to_int).values

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))

# SKLearn metrics
print("Confusion matrix:\n", cm)

# Balanced accuracy
bal_acc = balanced_accuracy_score(y_true, y_pred)
print("Balanced accuracy:", bal_acc)

# Optionally, show with class labels
import pandas as pd
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("\nConfusion Matrix (with labels):")
print(cm_df)


Confusion matrix:
 [[ 1334   106]
 [  666 17802]]
Balanced accuracy: 0.9451632553606237

Confusion Matrix (with labels):
             not-sitting  sitting
not-sitting         1334      106
sitting              666    17802


In [24]:
# animesh code
import numpy as np
compute_accuracy_from_confusion_matrix(cm)

(0.8812626027609741, 0.8836404273683072)

In [25]:
compute_additional_metrics_from_confusion_matrix(cm)

{'sensitivity': [0.8891173245614035, 0.8781635301752109],
 'specificity': [0.8781635301752109, 0.8891173245614035],
 'positive_predictive_value': [0.7422196796338673, 0.952545753167527],
 'negative_predictive_value': [0.952545753167527, 0.7422196796338673],
 'f1_score': [0.8090546270890497, 0.9138435565559933]}

In [8]:
import pandas as pd

#animesh_old = "/Users/leo/Desktop/DeepPostures_MAE/support_files/submit_result/H/CHAP-ZS/predictions/i0018A.csv"
animesh_old  = '/Users/leo/Desktop/DeepPostures_MAE/support_files/animesh CHAP-ZS/prediction_H/i0018A.csv'
animesh_new = "/Users/leo/Desktop/DeepPostures_MAE/support_files/submit_result/H/CHAP-ZS/predictions/i0018A.csv"
 #'/Users/leo/Desktop/DeepPostures_MAE/support_files/animesh CHAP-ZS/prediction_H/i0018A.csv'

# Load the CSVs
df_old = pd.read_csv(animesh_old)
df_new = pd.read_csv(animesh_new)

# Standardize timestamp column
df_old.rename(columns={'Timestamp': 'timestamp'}, inplace=True)
df_new.rename(columns={'Timestamp': 'timestamp'}, inplace=True)

# Perform inner join on timestamp
df_merged = pd.merge(df_old, df_new, on='timestamp', how='inner', suffixes=('_old', '_new'))

# Print column names to verify
print("Columns after merge:", df_merged.columns.tolist())

# Now compare the columns — update names if needed
label_col_old = 'label_old'
label_col_new = 'label_new'
pred_col_old = 'prediction_old'
pred_col_new = 'prediction_new'

if label_col_old in df_merged and label_col_new in df_merged:
    label_match = df_merged[label_col_old] == df_merged[label_col_new]
    print(f"Label match rate: {label_match.sum()}/{len(label_match)} ({label_match.mean():.2%})")

if pred_col_old in df_merged and pred_col_new in df_merged:
    pred_match = df_merged[pred_col_old] == df_merged[pred_col_new]
    print(f"Prediction match rate: {pred_match.sum()}/{len(pred_match)} ({pred_match.mean():.2%})")

    # Print mismatches
    mismatches = df_merged[~pred_match]
    print("Mismatched predictions:")
    print(mismatches[['timestamp', pred_col_old, pred_col_new]])
else:
    print("Could not find expected prediction columns.")


Columns after merge: ['segment_old', 'timestamp', 'label_old', 'prediction_old', 'segment_new', 'prediction_new', 'label_new']
Label match rate: 25404/25404 (100.00%)
Prediction match rate: 25034/25404 (98.54%)
Mismatched predictions:
                 timestamp prediction_old prediction_new
624    2013-05-07 15:17:00        sitting    not-sitting
625    2013-05-07 15:17:10        sitting    not-sitting
626    2013-05-07 15:17:20        sitting    not-sitting
627    2013-05-07 15:17:30        sitting    not-sitting
628    2013-05-07 15:17:40        sitting    not-sitting
...                    ...            ...            ...
25335  2013-05-15 12:40:30    not-sitting        sitting
25356  2013-05-15 12:44:00    not-sitting        sitting
25357  2013-05-15 12:44:10    not-sitting        sitting
25358  2013-05-15 12:44:20    not-sitting        sitting
25359  2013-05-15 12:44:30    not-sitting        sitting

[370 rows x 3 columns]
